****PRACTICA FINAL IA (YouTube Trending Videos)****

**Preprocesamiento**

In [1]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import col, lit, explode, max as _max, to_date, to_timestamp, year, month, trim, replace, split, array_remove,date_format
from functools import reduce
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
if os.name == 'nt':  # 'nt' significa que el sistema operativo es Windows
    os.environ["HADOOP_HOME"] = os.getcwd()
    os.environ["hadoop.home.dir"] = os.getcwd()

In [3]:
spark=SparkSession.builder.appName("Preprocesamiento").getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")

Actualizamos la categoria de nuestro fichero csv en base a los datos que tenemos en el fichero JSON.

In [4]:
def actualizacion_categoria_pais(ruta_csv, ruta_json, nombre_pais):

    """Función que actualiza la category_id del Csv reemplazando por el valor string asignado en el fichero json y añade columna con el pais."""
    
    dfjson = spark.read.option("multiLine", "true").json(ruta_json)#en vez de leer linea por linea, empaqueta todo el json
    dfjson_aplanado = dfjson.select(explode("items").alias("item")).select(col("item.id").alias("json_id"), col("item.snippet.title").alias("category"))
    
    dfcsv = spark.read.options(header=True,encoding='UTF-8').csv(ruta_csv)
    dfcsv = dfcsv.withColumn("category_id", col("category_id").cast("string"))
    
    df_final = dfcsv.join(dfjson_aplanado, dfcsv["category_id"] == dfjson_aplanado["json_id"], "left")
    
    df_final = df_final.drop("json_id","category_id")

    df_final = df_final.withColumn("pais", lit(nombre_pais))
                
    return df_final

In [5]:
paises = {"CA":"Canadá","DE":"Alemania","FR":"Francia","GB":"Gran Bretaña","IN":"India","JP":"Japón","KR":"Corea","MX":"Mexico","RU":"Rusia","US":"USA"}

In [6]:
dataframes_spark = (actualizacion_categoria_pais(f"{c}videos.csv", f"{c}_category_id.json", nombre) for c, nombre in paises.items())

Unificamos todos los dataframes

In [7]:
df_spark_unido= reduce(DataFrame.union, dataframes_spark)

Confirmamos que en nuestro dataframe final hay registros de todos los paises.

In [8]:
df_spark_unido.groupBy("pais").count().show()

+------------+-----+
|        pais|count|
+------------+-----+
|      Canadá|45560|
|    Alemania|46957|
|     Francia|46138|
|Gran Bretaña|43295|
|       India|38528|
|       Japón|21445|
|       Corea|36730|
|      Mexico|43819|
|       Rusia|46260|
|         USA|48137|
+------------+-----+



title: El título del video en formato de texto original libre (tal y como lo escribió el creador, incluyendo emojis, mayúsculas y caracteres especiales). 

channel_title: El nombre original del canal de YouTube que subió el video.  

category_id: El identificador de la categoría asignada al video. Actualizado más arriba por el país en cuestión. 

tags: Las palabras clave asociadas al video. 

views: La cantidad de reproducciones acumuladas que registraba el video en ese día específico en que apareció en tendencias.  

likes: La cantidad de votos "Me gusta" acumulados que tenía el video en ese día específico.  

dislikes: La cantidad de votos "No me gusta" acumulados que tenía el video en ese día específico.  

comment_count: El número de comentarios totales escritos por los usuarios en ese día específico.  

comments_disabled: Un interruptor lógico (True/False) que indica si el canal tenía los comentarios desactivados en ese momento.  

ratings_disabled: Un interruptor lógico (True/False) que indica si el canal tenía ocultos los contadores de likes/dislikes en ese momento.  

video_error_or_removed: Un interruptor lógico (True/False) que indica si el video fue borrado o tuvo un error mientras estaba en la lista de tendencias.  

description: El texto explicativo original largo que el creador colocó en la caja de descripción del video. 

pais: El país donde se obtuvo el dato.

In [9]:
print((df_spark_unido.count(), len(df_spark_unido.columns)))

(416869, 17)


Procedemos a quitar los duplicados.

In [10]:
df_spark_unido = df_spark_unido.dropDuplicates()

In [11]:
print((df_spark_unido.count(), len(df_spark_unido.columns)))

(369783, 17)


Se eliminan alrededor de 50000 duplicados

Estos son los items que tendremos que predecir:  
1.Prediga la categoría del vídeo  
2.Predecir el número de likes  
3.Encontrar grupos en los vídeos mediante diversas técnicas de clusterización  
4.Crear un recomendador 

Las columnas video_id, description y thumbnail_link  no tiene ningún tipo de información que nos pueda ayudar. Las eliminamos.

In [12]:
df_spark_unido = df_spark_unido.drop("video_id", "thumbnail_link","description")

In [13]:
df_spark_unido.limit(5).toPandas()

,trending_date,title,channel_title,publish_time,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed,category,pais
0,17.17.11,The Problem with Lab Mice,CollegeHumor,2017-11-15T18:30:00.000Z,"""Collegehumor""|""CH originals""|""comedy""|""sketch...",655161,21013,2563,2671,False,False,False,Comedy,Canadá
1,17.18.11,Equipe d'Algérie : Rabah Madjer - Maamar Djebb...,ALGERIE Football,2017-11-14T23:44:28.000Z,"""football""|""usmalger""|""mcalger""|""crbélouizdad""...",90474,101,22,85,False,False,False,Sports,Canadá
2,17.18.11,StarCraft II Free to Play: Legacy,StarCraft,2017-11-14T20:46:23.000Z,"""StarCraft""|""SC2""|""Terran""|""Zerg""|""Protoss""|""B...",710476,12395,244,589,False,False,False,Gaming,Canadá
3,17.19.11,【FULL】王俊凯古装上阵变身小皇子演技大爆发 章子怡惊艳旗袍造型对战周一围上演虐恋《演员的...,中国浙江卫视官方频道 Zhejiang TV Official Channel - 欢迎订阅 -,2017-11-18T18:07:05.000Z,"""演员的诞生""|""演员的诞生 整片""|""演员的诞生 正片""|""演员的诞生 4""|""演员的诞生...",161110,717,75,902,False,False,False,Entertainment,Canadá
4,17.19.11,Jimmy Fallon Pays Tribute to His Mother Gloria,The Tonight Show Starring Jimmy Fallon,2017-11-14T04:55:24.000Z,"""Jimmy Fallon""|""Tonight Show Starring Jimmy Fa...",2742344,89376,1373,5077,False,False,False,Comedy,Canadá


In [14]:
df_spark_unido.summary("count").toPandas()

,summary,trending_date,title,channel_title,publish_time,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed,category,pais
0,count,364873,364432,364175,363823,363563,363515,363486,363471,363458,363438,363424,363418,360662,369783


Tenemos 369783 registros. Vemos que en la mayoria de columnas no se llega a esta cantidad, probablemente por los nulos. Quitamos nulos.

In [15]:
df_spark_unido = df_spark_unido.dropna()

In [16]:
print((df_spark_unido.count(), len(df_spark_unido.columns)))

(360660, 14)


In [17]:
df_spark_unido.summary("count").toPandas()#Confirmamos que se han quitado todos los nulos.

,summary,trending_date,title,channel_title,publish_time,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed,category,pais
0,count,360660,360660,360660,360660,360660,360660,360660,360660,360660,360660,360660,360660,360660,360660


Vamos con title y channel title. Vemos que es un video de tendencias por lo que podria suceder que el mismo video fuera tendencia varios dias 
y por tantos los datos pudieran estar duplicados pero el drop_duplicates no haya funcionado porque las variables numéricas sean diferentes.

In [18]:
print(df_spark_unido.count() - df_spark_unido.select("title", "channel_title").distinct().count())

176006


Vemos que en las filas title y channel_title hay 170694 valores duplicados (tenemos 342274 valores en total). Vamos a intentar buscar un ejemplo para visualizarlo.

In [19]:
df_lista_duplicados = df_spark_unido.groupBy("title", "channel_title").count().filter("count > 1")
df_resultado_duplicados = df_spark_unido.join(df_lista_duplicados, ["title", "channel_title"], "inner").drop("count")
df_resultado_duplicados.limit(10).toPandas()

,title,channel_title,trending_date,publish_time,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed,category,pais
0,KAJAL RAGHWANI SUPERHIT MOVIE ( Full HD 2018...,Filmistaan,18.01.01,2017-12-29T13:48:36.000Z,"""kajal|""""kajal raghwani""""|""""raghwani kajal""""|""...",526046,668,313,79,FALSE,FALSE,FALSE,Film & Animation,India
1,KAJAL RAGHWANI SUPERHIT MOVIE ( Full HD 2018...,Filmistaan,17.31.12,2017-12-29T13:48:36.000Z,"""kajal|""""kajal raghwani""""|""""raghwani kajal""""|""...",443397,595,276,68,FALSE,FALSE,FALSE,Film & Animation,India
2,!SOY UN AZTECO!,Facundo,18.21.04,2018-04-19T20:54:04.000Z,"""facundo|""""humor""""|""""chistes""""|""""videos cortos...",425793,18930,754,1824,FALSE,FALSE,FALSE,Comedy,Mexico
3,!SOY UN AZTECO!,Facundo,18.22.04,2018-04-19T20:54:04.000Z,"""facundo|""""humor""""|""""chistes""""|""""videos cortos...",500922,20879,865,1902,FALSE,FALSE,FALSE,Comedy,Mexico
4,!SOY UN AZTECO!,Facundo,18.20.04,2018-04-19T20:54:04.000Z,"""facundo|""""humor""""|""""chistes""""|""""videos cortos...",221225,12450,381,1227,FALSE,FALSE,FALSE,Comedy,Mexico
5,"#10 Kappsägenstand | Schubladen bauen, Topfsc...",Lets-Bastel,18.22.03,2018-03-20T18:00:02.000Z,"""Holzwerken""|""Lets Bastel""|""Heimwerken""|""Bauan...",31677,853,79,240,False,False,False,Entertainment,Alemania
6,"#10 Kappsägenstand | Schubladen bauen, Topfsc...",Lets-Bastel,18.21.03,2018-03-20T18:00:02.000Z,"""Holzwerken""|""Lets Bastel""|""Heimwerken""|""Bauan...",22713,715,66,209,False,False,False,Entertainment,Alemania
7,#1023 Chopper & Usopp try to save Robin - Stra...,Abood Abuzarqa1,18.10.06,2018-06-10T13:56:09.000Z,"""anime|""""animation""""|""""SUB""""|""""HD""""|""""luffy""""|...",208369,951,54,247,FALSE,FALSE,FALSE,Film & Animation,Japón
8,#1023 Chopper & Usopp try to save Robin - Stra...,Abood Abuzarqa1,18.11.06,2018-06-10T13:56:09.000Z,"""anime|""""animation""""|""""SUB""""|""""HD""""|""""luffy""""|...",385948,1473,117,421,FALSE,FALSE,FALSE,Film & Animation,Japón
9,#1023 Chopper & Usopp try to save Robin - Stra...,Abood Abuzarqa1,18.10.06,2018-06-10T13:56:09.000Z,"""anime|""""animation""""|""""SUB""""|""""HD""""|""""luffy""""|...",208337,950,54,247,FALSE,FALSE,FALSE,Film & Animation,Japón


In [20]:
df_ejemplo = df_spark_unido.filter((col("title") == "!SOY UN AZTECO!") & (col("channel_title") == "Facundo"))
df_ejemplo.toPandas()

,trending_date,title,channel_title,publish_time,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed,category,pais
0,18.21.04,!SOY UN AZTECO!,Facundo,2018-04-19T20:54:04.000Z,"""facundo|""""humor""""|""""chistes""""|""""videos cortos...",425793,18930,754,1824,FALSE,FALSE,FALSE,Comedy,Mexico
1,18.20.04,!SOY UN AZTECO!,Facundo,2018-04-19T20:54:04.000Z,"""facundo|""""humor""""|""""chistes""""|""""videos cortos...",221225,12450,381,1227,FALSE,FALSE,FALSE,Comedy,Mexico
2,18.22.04,!SOY UN AZTECO!,Facundo,2018-04-19T20:54:04.000Z,"""facundo|""""humor""""|""""chistes""""|""""videos cortos...",500922,20879,865,1902,FALSE,FALSE,FALSE,Comedy,Mexico


Vemos que hay columnas repetidas en las que como pasa aquí el mismo mes fue trending en varias ocasiones. Haremos filtrado y dejaremos un valor único por país teniendo en cuenta principalmente la variable views, ya que a priori a mas alta dicha variable mas likes, dislikes y commments debería haber ya que el formato youtube va acumulando cada dia dichos items.

In [21]:
df_max_visitas = df_spark_unido.groupBy("title", "channel_title", "pais").agg(_max("views").alias("views"))
df_spark_unido = df_spark_unido.join(df_max_visitas, on=["title", "channel_title", "pais", "views"], how="inner")

In [22]:
print((df_spark_unido.count(), len(df_spark_unido.columns)))

(207745, 14)


In [ ]:
df_ejemplo = df_spark_unido.filter((col("title") == "!SOY UN AZTECO!") & (col("channel_title") == "Facundo"))
df_ejemplo.toPandas()

Comprobamos que ya se han quitado los duplicados y nos hemos quedado con los valores con mas views, de tal manera que nuestro dataset ha disminuido en casi 150000 registros.

In [ ]:
df_spark_unido.dtypes

Vemos que todas las columnas son string, modificamos el tipo de cada una

In [ ]:
df_spark_unido = df_spark_unido \
    .withColumn("views", col("views").cast("integer")) \
    .withColumn("likes", col("likes").cast("integer")) \
    .withColumn("dislikes", col("dislikes").cast("integer")) \
    .withColumn("comment_count", col("comment_count").cast("integer")) \
    .withColumn("comments_disabled", col("comments_disabled").cast("boolean")) \
    .withColumn("ratings_disabled", col("ratings_disabled").cast("boolean")) \
    .withColumn("video_error_or_removed", col("video_error_or_removed").cast("boolean")) \
    .withColumn("trending_date", date_format(to_date(col("trending_date"), "yy.dd.MM"), "dd/MM/yyyy")) \
    .withColumn("publish_time", date_format(col("publish_time").cast("date"), "dd/MM/yyyy"))

Cambiamos el formato de fecha

In [ ]:
df_spark_unido.dtypes#confirmamos el cambio de tipo

Vamos a modificar los formatos de cada columna. Comenzaremos con title de tipo string. Procedemos a quitar los espacios 
y los posibles saltos de línea al introducir el nombre del title.

In [ ]:
df_spark_unido = df_spark_unido \
    .withColumn("title", replace(col("title"), lit("\n"), lit(" "))) \
    .withColumn("title", replace(col("title"), lit("\t"), lit(" "))) \
    .withColumn("title", replace(col("title"), lit('"'), lit("")))

df_spark_unido = df_spark_unido.withColumn("title", trim(col("title")))

df_spark_unido = df_spark_unido \
    .withColumn("title", replace(col("title"), lit("  "), lit(" "))) \
    .withColumn("title", replace(col("title"), lit("  "), lit(" "))) \
    .withColumn("title", replace(col("title"), lit("  "), lit(" "))) #hacemos esto 3 veces con el fin de quitar lo máximos espacios posibles.

Realizamos la misma transformación en channel_title:

In [ ]:
df_spark_unido = df_spark_unido \
    .withColumn("channel_title", replace(col("channel_title"), lit("\n"), lit(" "))) \
    .withColumn("channel_title", replace(col("channel_title"), lit("\t"), lit(" "))) \
    .withColumn("channel_title", replace(col("channel_title"), lit('"'), lit("")))

df_spark_unido = df_spark_unido.withColumn("channel_title", trim(col("channel_title")))

df_spark_unido = df_spark_unido \
    .withColumn("channel_title", replace(col("channel_title"), lit("  "), lit(" "))) \
    .withColumn("channel_title", replace(col("channel_title"), lit("  "), lit(" "))) \
    .withColumn("channel_title", replace(col("channel_title"), lit("  "), lit(" "))) 

En el caso de tags además de los espacios en blanco, debemos de quitar "|" para que el modelo pueda leerlo mejor. 
Metemos los datos en una lista.

In [ ]:
df_spark_unido = df_spark_unido.withColumn("tags", replace(col("tags"), lit("|"), lit(" "))) \
    .withColumn("tags", replace(col("tags"), lit("\n"), lit(" "))) \
    .withColumn("tags", replace(col("tags"), lit("\t"), lit(" "))) \
    .withColumn("tags", replace(col("tags"), lit('"'), lit("")))

df_spark_unido = df_spark_unido.withColumn("tags", trim(col("tags")))

df_spark_unido = df_spark_unido \
    .withColumn("tags", replace(col("tags"), lit("  "), lit(" "))) \
    .withColumn("tags", replace(col("tags"), lit("  "), lit(" "))) \
    .withColumn("tags", replace(col("tags"), lit("  "), lit(" ")))

In [ ]:
df_spark_unido = df_spark_unido.withColumn("tags", array_remove(split(col("tags"), " "), "")) #usamos array_remove para quitar posibles espacios en blanco que quedaran tras el split

In [ ]:
df_spark_unido.limit(5).toPandas()

Limpio el dataset, lo guardamos en formato parquet (se optimiza el procesamiento utilizando este formato de almacenamiento de apache) y  pasamos al análisis exploratorio

In [ ]:
Ruta_limpia = ""
ruta_salida_final = f"{Ruta_limpia}Dataset_limpio.parquet"
df_spark_unido.toPandas().to_parquet(ruta_salida_final, index=False)

**Analisis exploratorio**

A continuación vamos a ver un poco los datos que tenemos después de procesar nuestro df. Para esta parte lo pasamos a pandas.

In [ ]:
df_spark_limpio = spark.read.parquet("Dataset_limpio.parquet")#Leemos el archivo creado anteriormente de su ruta
df = df_spark_limpio.toPandas() #pasamos para el EDA el archivo a Pandas

In [ ]:
df.info()

In [ ]:
df["trending_date"] = pd.to_datetime(df["trending_date"], format= "%d/%m/%Y")
df["publish_time"] = pd.to_datetime(df["publish_time"], format= "%d/%m/%Y")#pasamos a datetime ambas variables

In [ ]:
df.info()

In [ ]:
matriz_correlacion = df.corr(numeric_only=True)#sacamos la matriz de correlacion para vaer relacion de las variables numéricas

In [ ]:
print(matriz_correlacion)

Conclusiones:  

Likes y comment_count: Si hay un like, suele haber un comentario de manera generalizada.

Views y likes: A más visualizaciones más likes. Por lo tanto views, likes y comment_count suelen ir de la mano.

Dislikes con views, likes y comment_count: Cuanto más se ve el video, más posibles dislikes puede provocar y por tanto, mas comentarios a modo de hate.

In [ ]:
df.describe()

Podemos ver que los valores máximos y minimos distorsionan mucho con respecto a la media y si intentamos sacar outliers, vemos 
que hay en las variables views, likes, dislikes y comment_count. Esto es debido a la viralidad que se puede dar con cierto vídeo. Vamos a representar los outliers de views ya que como vimos anteriormente a partir de ella el resto de outliers que hemos visto también pueden aumentar dada su gran relación.

In [ ]:
Q1 = df["views"].quantile(0.25)
Q3 = df["views"].quantile(0.75)
IQR = Q3 - Q1 #Rango intercuantílico
limite_outliers = Q3 + (1.5 * IQR)
df_outliers = df[df["views"] > limite_outliers]

plt.figure(figsize=(4, 6))
sns.stripplot(y=df_outliers["views"], color="black", jitter=0.25,size=2)

plt.title("Outliers")
plt.ylabel("Views 1 = 100 millones")
plt.show()

Vemos claramente los outliers y de cara al EDA los dejamos ya que según que planteamiento puede ser bueno quitarlos o no.

**Analisis univariado de variables categoricas**

In [ ]:
df.category.value_counts(normalize=True)

Vemos que más de la mitad de los videos son de la categoría Entertainment, People & Blogs y News & Politics.

In [ ]:
df.category.value_counts(normalize=True).plot.barh()
plt.show()

In [ ]:
df.pais.value_counts(normalize=True).plot.barh()
plt.show()

Vemos que lo datos se centran sobre todo en los paises Rusia, Mexico, Alemania, Francia y Canada

In [ ]:
df['comments_disabled'].value_counts()

In [ ]:
df['ratings_disabled'].value_counts()

In [ ]:
df['video_error_or_removed'].value_counts()

In [ ]:
df.drop('video_error_or_removed', axis=1, inplace=True)

Vemos que no tiene ningún valor esta variable por lo que la eliminamos ahora aquí e insertamos la actualización el fichero preprocess.

In [ ]:
df.sort_values(by='views', ascending=False).head(10)

Vemos que hay videos que han sido tendencia en diferentes países.

**Analisis bivariado de variables categoricas**

In [ ]:
df.pivot_table(index="pais", columns="category", values="views")

In [ ]:
pivot_views = df.pivot_table(index="pais", columns="category", values="views")
plt.figure(figsize=(14, 7))

sns.heatmap(pivot_views, cmap="Blues", annot=False, linewidths=0.5)

plt.title("Views por país y categoría",fontsize=15)
plt.xlabel("Categorías de Video", fontsize=11)
plt.ylabel("Países", fontsize=11)
plt.xticks(rotation=45, ha="right") 

plt.tight_layout()
plt.show()

Vemos que las mayorias de visualizaciones se dan en Gran Bretaña y EEUU y que la categoría que más views suele tener es Music. El hecho de que un video sea de Gran Bretaña y de Musica puede ser relevante por ejemplo para el modelo, por lo que ambas categorías es bueno tenerlas en cuenta.

Vamos a desglosar comments_disabled	y ratings_disable. Estas dos variables la mayoría del tiempo son False ya que lo normal es querer que la gente comente y haga like o dislike a tus videos. Vamos a ver en cuantos casos desactivaron ambas variables.

In [ ]:
Desactivadas = df[(df["comments_disabled"] == True) & (df["ratings_disabled"] == True)]

In [ ]:
Desactivadas.shape

In [ ]:
Desactivadas = df[(df["comments_disabled"] == True) & (df["ratings_disabled"] == True)]
Activadas = df[(df["comments_disabled"] == False) & (df["ratings_disabled"] == False)]

promedio_desactivadas = Desactivadas[["views"]].mean()# Calculamos los promedios de 'views' y 'likes' para ambos grupos
promedio_activadas = Activadas[["views"]].mean()

# 4. Juntamos ambos resultados en una pequeña tabla para comparar de reojo
comparativa_directa = pd.DataFrame({"Ambas Activadas": promedio_activadas, "Ambas Desactivadas": promedio_desactivadas}).T #Transponemos para que se cambien columnas y filas

print(comparativa_directa)

Vemos que cuando los comentarios y los ratings estan deshabilitados, suele haber mas visualizaciones por parte de los usuarios. Normalmente este tipo de sucesos se dan porque los videos suelen ser polémicos y los creadores de contenido quitan los comentarios y las interacciones.

In [ ]:
datos = df.copy()
datos["dias_hasta_tendencia"] = (datos["trending_date"] - datos["publish_time"]).dt.days
print(datos[["dias_hasta_tendencia"]].describe())

Vemos que de media los videos suelen tardar unos 4 días en ser tendencia, aunque la mediana nos indica que de normal suele ser 1 día. La dispersión es grande y se puede ver porque por ejemplo el vídeo máximo tardo 4215 dias en ser tendencia algo poco común.

Las conclusiones que podemos sacar son las siguientes:
1. Existe una relación clara entre los views, los coments y los likes/dislikes.
2. Entertainment, People & Blogs y News & Politics son las categorías que más destacan.
3. Un mismo vídeo puede ser viral en varios países.
4. Existe una relación directamente proporcional entre bloquear los comentarios y los ratings y las visualizaciones.
5. Cruzando datos vemos que los videos que más views han tenido son de Musica fundamentalmente en Reino Unido. Tanto Reino Unido como el tipo Musica muestran ser una de las piezas fundamentales sobre las que giran las visualizaciones.

A continuación vamos a pasar a realizar punto por punto lo solicitado en los modelos.

**Predecir categoria del video**

Vamos a utilizar los modelos conocidos pero en primer lugar vamos a rescatar el dataset limpio en formato parquet (lo hemos estado trabajando en pandas para el EDA), con el fin de ganar mayor rendimiento con Spark ML

In [ ]:
df_spark_ml = spark.read.parquet("Dataset_limpio.parquet")

Regresión logística

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

In [ ]:
columnas_categoricas = ["category", "pais"]
indexer = StringIndexer(inputCols=columnas_categoricas, outputCols=[f"{c}_indexado" for c in columnas_categoricas], handleInvalid="keep")
#Hacemos el string indexer para las variables categóricas, category y pais, para poder usarlas en el modelo de regresión logística.

In [ ]:
encoder = OneHotEncoder(inputCols=["pais_indexado"],outputCols=["pais_encoded"])
#Hacemos el one hot encoder para la variable pais_indexado  , para poder usarla en el modelo de regresión logística. 
#No usamos category ya que es la variable objetivo

In [ ]:
#Cogemos las columnas predictoras y las juntamos en una sola columna llamada features, que es la que usará el modelo de regresión logística.
features = ["pais_encoded", "views", "likes", "dislikes", "comment_count", "comments_disabled", "ratings_disabled"]
Assembler = VectorAssembler(inputCols=features, outputCol="features")

In [ ]:
lr = LogisticRegression(featuresCol="features", labelCol="category_indexado"

In [ ]:
df_train, df_test = df_spark_ml.randomSplit([0.7, 0.3], seed=42)

In [ ]:
Pipeline = Pipeline(stages=[indexer, encoder, Assembler,lr])
Pipeline_model = Pipeline.fit(df_train)
df_transformado = Pipeline_model.transform(df_test)

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
#Al estar los datos de categorias desbalanceados, usamos la métrica weightedF1 para evaluar cual es mejor opción de los 
#modelos que usaremos de clasifiacion
evaluator = MulticlassClassificationEvaluator(labelCol="category_indexado",predictionCol="prediction",metricName="weightedF1")
weightedF1 = evaluator.evaluate(df_transformado)
print("weightedF1:", weightedF1)


Random Forest

In [ ]:

)

**Predecir el número de likes**

**Encontrar grupos en los vídeos mediante diversas técnicas de clusterización**

**Crear un recomendador**

""title: El título del video en formato de texto original libre (tal y como lo escribió el creador, incluyendo emojis, mayúsculas y caracteres especiales). 

channel_title: El nombre original del canal de YouTube que subió el video.  

category_id: El identificador de la categoría asignada al video. Actualizado más arriba por el país en cuestión. 

tags: Las palabras clave asociadas al video. 

views: La cantidad de reproducciones acumuladas que registraba el video en ese día específico en que apareció en tendencias.  

likes: La cantidad de votos "Me gusta" acumulados que tenía el video en ese día específico.  

dislikes: La cantidad de votos "No me gusta" acumulados que tenía el video en ese día específico.  

comment_count: El número de comentarios totales escritos por los usuarios en ese día específico.  

comments_disabled: Un interruptor lógico (True/False) que indica si el canal tenía los comentarios desactivados en ese momento.  

ratings_disabled: Un interruptor lógico (True/False) que indica si el canal tenía ocultos los contadores de likes/dislikes en ese momento.  

video_error_or_removed: Un interruptor lógico (True/False) que indica si el video fue borrado o tuvo un error mientras estaba en la lista de tendencias.  

description: El texto explicativo original largo que el creador colocó en la caja de descripción del video. 

pais: El país donde se obtuvo el dato.

Estos son los items que tendremos que predecir:  
1.Prediga la categoría del vídeo  
2.Predecir el número de likes  
3.Encontrar grupos en los vídeos mediante diversas técnicas de clusterización  
4.Crear un recomendador

[('title', 'string'), 
 ('channel_title', 'string'),
 ('pais', 'string'),
 ('views', 'int'),
 ('tags', 'string'),
 ('likes', 'int'),
 ('dislikes', 'int'),
 ('comment_count', 'int'),
 ('comments_disabled', 'boolean'),
 ('ratings_disabled', 'boolean'),
 ('video_error_or_removed', 'boolean'),
 ('description', 'string'),
 ('category', 'string'),
 ('month_trending', 'int'),
 ('month_publish', 'int')]"""
 • ¿Estás clasificando categorías? Usa Regresión Logística, Árboles de decisión, KNN o SVM.
• ¿Estás prediciendo un número? Usa Regresión Lineal o Árboles de decisión (Regresores).
• ¿Tus datos no tienen etiquetas (no sabes la respuesta)? Usa Clustering o Reducción de dimensionalidad.